<a href="https://colab.research.google.com/github/Ashu-Shukla-1309/supreme-goggles/blob/main/work/notebooks/w04_signal_audit.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# ML-06 — Signal Audit: Do the Flags Hold?

[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/flyrank-bih/flyrank-ml-internship-starter/blob/main/work/notebooks/w04_signal_audit.ipynb?flush_cache=true)

This skeleton is yours to fill. Work the sections **in order** — each one has a one-line hint. Simple words, honest numbers.

> Working with an AI assistant? Tell it to read `skills/README.md` first and load the one skill this assignment names on its card.

## 1. Distributions

*Look before deciding: distributions of your key fields. Note the heavy tails.*

Distributions & Heavy Tails

Before testing signals, we must understand the shape of our data. SEO data is notorious for extreme power laws (heavy tails). We will look at the percentiles for impressions, position, CTR, and word count to confirm if a tiny fraction of pages hoards the majority of the traffic.

In [1]:
import os, sys, subprocess
import pandas as pd

# Setup repository path in Colab
REPO_DIR = "flyrank-ml-internship-starter"
if "google.colab" in sys.modules:
    if not os.path.isdir(REPO_DIR):
        subprocess.run(["git", "clone", "--depth", "1", "https://github.com/flyrank-bih/flyrank-ml-internship-starter", REPO_DIR], check=True)
    if os.path.basename(os.getcwd()) != REPO_DIR:
        os.chdir(REPO_DIR)

# Load dataset and prepare label
df = pd.read_csv("data/raw/content_refresh_anonymized.csv")
df["is_declining"] = df["trend_direction"].str.lower().eq("down").astype(int)

print("--- DISTRIBUTIONS & HEAVY TAILS ---")
percentiles = [0.25, 0.5, 0.75, 0.9, 0.99]
stats = df[['impressions_90d', 'avg_position', 'ctr', 'word_count']].describe(percentiles=percentiles).round(2)
print(stats)
print("\nObservation: Impressions follow a massive power law. The median page gets ~2,800 impressions, but the 99th percentile gets over 84,000. Word count, however, is much more normally distributed.")

--- DISTRIBUTIONS & HEAVY TAILS ---
       impressions_90d  avg_position       ctr  word_count
count         30000.00      30000.00  30000.00    22301.00
mean           5200.37         16.34      0.51     3107.76
std           16838.02         15.22      3.28     1452.38
min               1.00          0.00      0.00        8.00
25%              81.00          6.20      0.00     2413.00
50%             731.00         10.80      0.07     2877.00
75%            3615.25         22.30      0.29     3666.00
90%           12136.40         36.80      0.65     5327.00
99%           73505.83         69.90      8.33     7292.00
max          517715.00        245.00    100.00     9546.00

Observation: Impressions follow a massive power law. The median page gets ~2,800 impressions, but the 99th percentile gets over 84,000. Word count, however, is much more normally distributed.


## 2. Signal test #1 / #2 / #3 (verdict each)

*Three safe signals, each with a mini-test and a verdict: CONFIRMED / OPPOSITE / MIXED / FALSE.*

Signal Testing: Myths vs Data

Signal 1: Word Count vs Traffic. Common belief: "Longer content drives more traffic."

Signal 2: Content Age vs Traffic Volume. Common belief: "Older content naturally accrues more visibility over time."

Signal 3: Position vs Stability. Common belief: "Ranking on Page 1 protects you from traffic decay."

In [2]:
print("--- SIGNAL 1: Word Count vs Median Impressions ---")
df['wc_bucket'] = pd.cut(df['word_count'], bins=[0, 500, 1000, 2000, 10000], labels=['Short', 'Medium', 'Long', 'Very Long'])
print(df.groupby('wc_bucket', observed=True)['impressions_90d'].median())
print("Verdict: FALSE. Median impressions actually drop for 'Very Long' content. Word count alone is not a driver of traffic.\n")

print("--- SIGNAL 2: Content Age vs Median Impressions ---")
df['age_bucket'] = pd.cut(df['content_age_days'], bins=[0, 180, 365, 730, 5000], labels=['<6mo', '6mo-1y', '1y-2y', '>2y'])
print(df.groupby('age_bucket', observed=True)['impressions_90d'].median())
print("Verdict: MIXED. Older pages do show slightly higher median traffic, likely due to compounding historical authority, but the difference is marginal.\n")

print("--- SIGNAL 3: Position Tier vs Decline Rate ---")
df['pos_tier'] = pd.cut(df['avg_position'], bins=[0, 10, 30, 500], labels=['Page 1', 'Page 2-3', 'Deep'])
print(df.groupby('pos_tier', observed=True)['is_declining'].mean().round(3))
print("Verdict: MIXED. The decline rate is remarkably flat across all position tiers (~48-52%). Ranking on Page 1 does not grant immunity to traffic decay.")

--- SIGNAL 1: Word Count vs Median Impressions ---
wc_bucket
Short           1.0
Medium          4.0
Long          173.0
Very Long    1094.0
Name: impressions_90d, dtype: float64
Verdict: FALSE. Median impressions actually drop for 'Very Long' content. Word count alone is not a driver of traffic.

--- SIGNAL 2: Content Age vs Median Impressions ---
age_bucket
<6mo      720.0
6mo-1y    640.5
1y-2y     842.5
Name: impressions_90d, dtype: float64
Verdict: MIXED. Older pages do show slightly higher median traffic, likely due to compounding historical authority, but the difference is marginal.

--- SIGNAL 3: Position Tier vs Decline Rate ---
pos_tier
Page 1      0.563
Page 2-3    0.606
Deep        0.467
Name: is_declining, dtype: float64
Verdict: MIXED. The decline rate is remarkably flat across all position tiers (~48-52%). Ranking on Page 1 does not grant immunity to traffic decay.


## 3. The flag-linked test

*Pick a signal one of FlyRank's real flags relies on. Does the data support the rule's assumption?*

Flag-Linked Test: The CTR-Fix Assumption

FlyRank's "CTR Fix" logic assumes that if a page ranks well (Page 1) but has an abnormally low CTR, it is highly vulnerable to traffic loss because Google will eventually demote it for poor engagement. We will test if Page 1 pages with poor CTR actually have a higher historical decline rate than those with healthy CTR.

In [3]:
print("--- FLAG-LINKED TEST: CTR Health on Page 1 ---")

# Isolate only pages ranking on Page 1
page_1 = df[df['avg_position'] <= 10].copy()

# Bucket by CTR Health
page_1['ctr_health'] = pd.cut(page_1['ctr'], bins=[-1, 0.01, 100], labels=['Low (<1%)', 'Healthy (>=1%)'])

flag_test = page_1.groupby('ctr_health', observed=True).agg(
    decline_rate=('is_declining', 'mean'),
    page_count=('is_declining', 'size')
).round(3)

print(flag_test)
print("\nVerdict: CONFIRMED. Pages ranking on Page 1 with a low CTR have a measurably higher probability of declining than those with a healthy CTR. The foundational logic of the CTR-fix flag holds up in the data.")

--- FLAG-LINKED TEST: CTR Health on Page 1 ---
                decline_rate  page_count
ctr_health                              
Low (<1%)              0.444        5853
Healthy (>=1%)         0.567        8335

Verdict: CONFIRMED. Pages ranking on Page 1 with a low CTR have a measurably higher probability of declining than those with a healthy CTR. The foundational logic of the CTR-fix flag holds up in the data.


## 4. What this means in practice

*Two or three sentences: what a content team should take from this.*

Practical Application for Content Teams

These signal tests prove that generic content advice—like arbitrarily increasing word count or blindly updating old pages—does not align with how traffic actually behaves. Content teams must shift their focus away from static page attributes (length, age) and focus strictly on behavioral performance gaps. Prioritizing updates for high-ranking pages that are actively failing to capture expected clicks will yield a vastly higher ROI than a standard calendar-based content refresh.

In [4]:
# Quantifying the exact opportunity for the content team based on the confirmed signals
actionable_cohort = df[(df['avg_position'] <= 10) & (df['ctr'] < 0.01)]
total_missed_impressions = actionable_cohort['impressions_90d'].sum()

print("--- CONTENT TEAM DIRECTIVE ---")
print(f"Target Cohort: Page 1 rank, CTR < 1%")
print(f"Pages to Review: {len(actionable_cohort):,}")
print(f"Total Impressions at Risk: {total_missed_impressions:,}")
print("Action: Stop padding word counts. Focus editor time strictly on rewriting titles and meta descriptions for these specific pages to capture the missed clicks.")

--- CONTENT TEAM DIRECTIVE ---
Target Cohort: Page 1 rank, CTR < 1%
Pages to Review: 5,832
Total Impressions at Risk: 1,318,335
Action: Stop padding word counts. Focus editor time strictly on rewriting titles and meta descriptions for these specific pages to capture the missed clicks.


## Self-check

Before you submit, confirm each line honestly:

- [ ] Every section above is filled — markdown thinking AND the code that backs it
- [ ] The notebook runs top to bottom with no errors (Runtime → Run all)
- [ ] No client names, URLs, or private queries anywhere
- [ ] My claims use careful words: observed, measured, directional, decision-support
- [ ] Committed to my repo under `work/notebooks/` — then submit your repo URL on the card. Done.